# Attention figure (t2pmhc-gcn) — individual panels a–g

Each panel is rendered and saved individually to `figures/attention/`, reproducing the model-attention
figure from the vendored tables in `data/attention/`:
- **a** normalized attention per domain, Binders/Negatives, by binding-probability bin
- **b** Δ-attention (binder − non-binder) over the top-5 peptides + per-peptide AUC
- **c** median attention per domain for GILGFVFTL
- **d/e** A\*02:01 (9-mer) / B\*08:01 (8-mer) sequence logo + per-position attention (binders)
- **f** normalized node-importance heatmap, allele × peptide position (anchor triangles)
- **g** CDR3α/CDR3β mean-contacts vs mean-attention, with Spearman ρ

B\*08:01 uses its 8-mer table for the panel-e logo, and the main (9-mer) table for panel f.
Run from `publication/notebooks/`. Reads only from `data/attention/`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
import seaborn as sns
import logomaker
from scipy.stats import spearmanr

pd.set_option("display.max_columns", None)
PUB = Path.cwd().parent
sys.path.insert(0, str(PUB / "lib"))
from figure_style import apply_style, figsize, save_figure
from benchmarker import Benchmarker

apply_style()
bmarker = Benchmarker()
DATA, RESULTS, FIGURES = PUB / "data", PUB / "results", PUB / "figures"
ATT = DATA / "attention"

## Load data + helpers

In [ ]:
mean_df = pd.read_csv(ATT / "mean_df.tsv", sep="\t")                       # panels a/b/c
pos_att = pd.read_csv(ATT / "peptide_position_attention.tsv", sep="\t")    # panels d/f/g (9-mer B*08:01)
b08_att = pd.read_csv(ATT / "b0801_position_attention.tsv", sep="\t")      # panel e (8-mer B*08:01)
aa_bg = pd.read_csv(ATT / "aa_background.tsv", sep="\t")                    # logo IC background

DOMAIN_ORDER = ["cdr3a", "cdr3b", "mhc", "peptide", "tcra", "tcrb"]
DOMAIN_LABELS = {"cdr3a": "CDR3α", "cdr3b": "CDR3β", "mhc": "MHC",
                 "peptide": "Peptide", "tcra": "TCRα", "tcrb": "TCRβ"}
AA = list("ACDEFGHIKLMNPQRSTVWY")
BACKGROUND = aa_bg.set_index("aa").loc[AA, "freq"].to_numpy()

# alleles whose per-allele total (both binders) exceeds 100 samples -> panels d/e/f
_allele_n = pos_att.groupby(["allele", "binder"])["n"].first().groupby("allele").sum()
GATED_ALLELES = sorted(_allele_n[_allele_n > 100].index)

# input-shape checks
assert set(mean_df["complex"]) == set(DOMAIN_ORDER), mean_df["complex"].unique()
assert {"complex", "att_mean", "binder", "peptide", "allele", "binder_prob"} <= set(mean_df.columns)
assert {"allele", "binder", "position", "mean_attention", "n"} <= set(pos_att.columns)
assert len(BACKGROUND) == 20 and abs(BACKGROUND.sum() - 1) < 0.05
assert len(GATED_ALLELES) == 9, GATED_ALLELES
assert b08_att["position"].max() == 8, "panel-e table must be 8-mer"
print("gated alleles:", GATED_ALLELES)


def relabel_domains(ax):
    ax.set_xticks(range(len(DOMAIN_ORDER)))
    ax.set_xticklabels([DOMAIN_LABELS[d] for d in DOMAIN_ORDER], rotation=30, ha="right")


def allele_code(allele):
    return allele.replace("*", "").replace(":", "")


def read_ligand(allele):
    lf = ATT / "ligand" / f"data_classI_MS_Peptides_{allele_code(allele)}.txt"
    return [x.strip() for x in open(lf) if x.strip()]


def get_logoplot_df(peptides, L, background, threshold=1):
    """Information-content (bits) matrix for a logo + the top-2 anchor positions (IC >= threshold)."""
    counts = np.zeros((L, len(AA)))
    for pep in peptides:
        for i, aa in enumerate(pep):
            if aa in AA:
                counts[i, AA.index(aa)] += 1
    f_a = counts / counts.sum(axis=1, keepdims=True)
    p_a = f_a / background
    p_a = p_a / p_a.sum(axis=1, keepdims=True)
    IC = np.log2(20) + np.sum(p_a * np.log2(p_a + 1e-20), axis=1)
    heights = p_a * IC[:, None]
    df = pd.DataFrame(heights, columns=AA)
    cand = sorted([(i, v) for i, v in enumerate(IC) if v >= threshold],
                  key=lambda x: x[1], reverse=True)[:2]
    return df, [i for i, _ in cand]

## Panel a — attention per domain by binding-probability bin

In [ ]:
fig, (ax_b, ax_n) = plt.subplots(1, 2, figsize=figsize("double", 85), sharey=True,
                                 constrained_layout=True)
for ax, binder, title in zip([ax_b, ax_n], [1, 0], ["Binders", "Negatives"]):
    df = mean_df[mean_df["binder"] == binder].copy()
    df["prob_bin"] = pd.cut(df["binder_prob"], bins=5,
                            labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"],
                            include_lowest=True)
    sns.boxplot(df, x="complex", y="att_mean", hue="prob_bin", showfliers=False,
                palette="coolwarm", order=DOMAIN_ORDER, ax=ax)
    ax.set_xlabel("Domain")
    ax.set_title(title)
    relabel_domains(ax)
ax_b.set_ylabel("Normalized Attention")
if ax_b.get_legend():
    ax_b.get_legend().remove()
ax_n.set_ylabel("")
ax_n.tick_params(labelleft=False)
sns.move_legend(ax_n, "upper left", bbox_to_anchor=(1.02, 1.0), title="Binding Probability", frameon=False)
sns.despine(ax=ax_b); sns.despine(ax=ax_n)
save_figure(fig, "panel_a_domain_prob_bins", subdir="attention")
plt.show()

## Panel b — Δ-attention heatmap + per-peptide AUC

In [ ]:
top5 = mean_df["peptide"].value_counts().head(5).index.tolist()
top5_df = mean_df[mean_df["peptide"].isin(top5)]
b1, b0 = top5_df[top5_df.binder == 1], top5_df[top5_df.binder == 0]
diff = (b1.groupby(["peptide", "complex"])["att_mean"].mean()
        - b0.groupby(["peptide", "complex"])["att_mean"].mean()).reset_index(name="delta_att")
auc_values = (top5_df.groupby("peptide")
              .apply(lambda g: bmarker.get_auc_df(g, "gcn"), include_groups=False)
              .reset_index(name="auc"))
sorted_pep = auc_values.sort_values("auc", ascending=False)["peptide"]
pivot = diff.pivot(index="peptide", columns="complex", values="delta_att")[DOMAIN_ORDER].loc[sorted_pep]
auc_sorted = auc_values.set_index("peptide").loc[sorted_pep]
assert list(pivot.index) == list(sorted_pep) and pivot.shape == (5, 6)

fig = plt.figure(figsize=figsize("double", 80), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[2, 0.75, 0.12], wspace=0.05)
ax0, ax1, cax = fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2])
hm = sns.heatmap(pivot, cmap="coolwarm", center=0, linewidths=0.6, linecolor="white", cbar=False, ax=ax0)
ax0.set_xticklabels([DOMAIN_LABELS[t.get_text()] for t in ax0.get_xticklabels()], rotation=45, ha="right")
ax0.set_xlabel("Domain"); ax0.set_ylabel("Peptide")
for s in ["top", "right"]:
    ax0.spines[s].set_visible(False)
sns.barplot(y=auc_sorted.index, x=auc_sorted["auc"], color="#6e6e6e", ax=ax1)
ax1.set_xlim(0, 1); ax1.set_xlabel("AUC"); ax1.set_ylabel("")
ax1.tick_params(left=False, labelleft=False)
for s in ["top", "right", "left"]:
    ax1.spines[s].set_visible(False)
fig.colorbar(hm.collections[0], cax=cax, label="Δ attention (binder − non-binder)")
cax.tick_params(length=0)
save_figure(fig, "panel_b_delta_auc", subdir="attention")
plt.show()

## Panel c — GILGFVFTL attention per domain

In [ ]:
fig, ax = plt.subplots(figsize=figsize("double", 75), constrained_layout=True)
gilg = mean_df[mean_df["peptide"] == "GILGFVFTL"]
sns.pointplot(data=gilg, x="complex", y="att_mean", hue="binder", hue_order=[1, 0],
              estimator="median", errorbar=("pi", 95), linestyle="none", dodge=0.4,
              markers=["s", "o"], palette=["#D86020", "black"], order=DOMAIN_ORDER, ax=ax)
ax.set_ylabel("Normalized Attention"); ax.set_xlabel("Domain")
ax.set_yticks(np.arange(0, 0.6, 0.1))
relabel_domains(ax); sns.despine(ax=ax)
handles = [Line2D([0], [0], marker="s", linestyle="None", color="#D86020", label="Binder", markersize=8),
           Line2D([0], [0], marker="o", linestyle="None", color="black", label="Non-binder", markersize=8)]
ax.legend(handles=handles, title="Binder status", frameon=True, loc="upper right")
ax.set_title("GILGFVFTL")
save_figure(fig, "panel_c_gilg", subdir="attention")
plt.show()

## Panels d/e — sequence logo + per-position attention (binders)

In [ ]:
def make_logo_panel(allele, L, att_df, name):
    peptides = [p for p in read_ligand(allele) if len(p) == L]
    motif_df, _ = get_logoplot_df(peptides, L, BACKGROUND, threshold=1)
    fig = plt.figure(figsize=figsize("double", 120), constrained_layout=True)
    gs = fig.add_gridspec(2, 1, height_ratios=[2, 1], hspace=0.05)
    ax_logo, ax_att = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    logomaker.Logo(motif_df, ax=ax_logo, color_scheme="chemistry",
                   show_spines=True, shade_below=False, fade_below=False)
    ax_logo.set_ylabel("Bits"); ax_logo.set_ylim(0, 2.5); ax_logo.set_yticks(np.arange(0, 2.6, 0.5))
    ax_logo.set_xticks([]); ax_logo.set_title(allele, fontweight="bold")
    for s in ["top", "right"]:
        ax_logo.spines[s].set_visible(False)
    sub = att_df[(att_df.allele == allele) & (att_df.binder == 1)].sort_values("position")
    attention = sub["mean_attention"].to_numpy()[:L]
    assert len(attention) == L, (allele, len(attention))
    ax_att.bar(np.arange(L), attention, color="#d98b5d", width=0.7)
    ax_att.set_ylabel("Attention"); ax_att.set_xlabel("Position")
    ax_att.set_xticks(np.arange(L)); ax_att.set_xticklabels(np.arange(1, L + 1))
    for s in ["top", "right"]:
        ax_att.spines[s].set_visible(False)
    save_figure(fig, name, subdir="attention")
    plt.show()

make_logo_panel("A*02:01", 9, pos_att, "panel_d_logo_a0201")   # 9-mer
make_logo_panel("B*08:01", 8, b08_att, "panel_e_logo_b0801")   # 8-mer

## Panel f — normalized node-importance heatmap (allele × position)

In [ ]:
binders = pos_att[(pos_att.binder == 1) & (pos_att.allele.isin(GATED_ALLELES))]
order = [a for a in GATED_ALLELES if a in binders["allele"].values]
mat = binders.pivot(index="allele", columns="position", values="mean_attention").loc[order]
mat_norm = mat.div(mat.max(axis=1), axis=0)

# per-allele anchors from the ligand-file logo at that allele's dominant peptide length
anchor_dict = {}
for allele in mat_norm.index:
    peps = read_ligand(allele)
    Lmode = int(pd.Series([len(p) for p in peps]).mode()[0])
    _, anchors = get_logoplot_df([p for p in peps if len(p) == Lmode], Lmode, BACKGROUND, threshold=1)
    anchor_dict[allele] = anchors

fig, ax = plt.subplots(figsize=figsize("double", 95), constrained_layout=True)
sns.heatmap(mat_norm, cmap="magma", cbar=True, linewidths=0.4, linecolor="black",
            ax=ax, cbar_kws={"label": "Normalized\nNode Importance"})
for row_idx, allele in enumerate(mat_norm.index):
    for a in anchor_dict[allele]:
        cx, cy, size = a + 0.5, row_idx + 0.5, 0.32
        ax.add_patch(patches.Polygon(
            [(cx, cy + size / 2), (cx - size / 2, cy - size / 2), (cx + size / 2, cy - size / 2)],
            closed=True, facecolor="none", edgecolor="white", linewidth=1.4))
ax.set_xlabel("Peptide Position"); ax.set_ylabel("Allele")
ax.set_xticklabels(range(1, mat_norm.shape[1] + 1), rotation=0)
for s in ax.spines.values():
    s.set_visible(False)
save_figure(fig, "panel_f_anchor_heatmap", subdir="attention")
plt.show()

## Panel g — CDR3 contacts vs attention (A\*02:01 binders)

In [ ]:
contact = pd.read_csv(ATT / "contact_files" / "a0201_binders_contacts.tsv", sep="\t")
contact["position"] = np.arange(1, len(contact) + 1)
sub = pos_att[(pos_att.allele == "A*02:01") & (pos_att.binder == 1)].sort_values("position")
n = int(sub["n"].iloc[0])
plot_df = pd.DataFrame({
    "position": contact["position"].to_numpy(),
    "attention": sub["mean_attention"].to_numpy(),
    "cdr3a": contact["cdr3a"].to_numpy() / n,
    "cdr3b": contact["cdr3b"].to_numpy() / n,
})

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=figsize("double", 85), sharex=True, sharey=True,
                                 constrained_layout=True)
gstats = {}
for ax, chain, label in zip([ax_a, ax_b], ["cdr3a", "cdr3b"], ["CDR3α", "CDR3β"]):
    sns.regplot(data=plot_df, x="attention", y=chain, ax=ax, color="black",
                line_kws={"lw": 1.5, "alpha": 0.8})
    for _, r in plot_df.iterrows():
        ax.text(r["attention"], r[chain], f"P{int(r['position'])}", fontsize=8, alpha=0.7, ha="right")
    rho, p = spearmanr(plot_df[chain], plot_df["attention"])
    gstats[chain] = (rho, p)
    ax.set_title(label)
    ax.text(0.05, 0.95, f"ρ={rho:.2f}", transform=ax.transAxes, va="top",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9))
    ax.set_xlabel("")
ax_a.set_ylabel("Mean CDR3 Contacts"); ax_b.set_ylabel("")
fig.supxlabel("Mean Attention Weight")
sns.despine(fig=fig)
save_figure(fig, "panel_g_cdr3_contacts", subdir="attention")
plt.show()
gstats

## Result tables

In [ ]:
RESULTS.mkdir(exist_ok=True)
bmarker.save_table(diff.merge(auc_values, on="peptide"), str(RESULTS / "attention_delta_auc.csv"))
gilg_med = (mean_df[mean_df["peptide"] == "GILGFVFTL"]
            .groupby(["complex", "binder"])["att_mean"].median().reset_index())
bmarker.save_table(gilg_med, str(RESULTS / "attention_gilg_domain_medians.csv"))
corr = pd.DataFrame([{"chain": c, "rho": r, "p": p} for c, (r, p) in gstats.items()])
bmarker.save_table(corr, str(RESULTS / "attention_cdr3_contact_correlation.csv"))
corr